# TrendPilot — Agentic AI Content Planning Agent

## AIRI Team PITB Internship Task 2

This notebook contains the complete local project workflow: setup, Ollama connection, content tools, structured agent, File Saver, SQLite memory, tests, five-output batch, Streamlit app generator, README, and report draft.

**Model:** `qwen2.5:3b` through Ollama  
**Interface:** Streamlit (local)  
**Important limitation:** TrendPilot generates content suggestions. It does not access live social-media trend data unless a real trend source is added.

### Recommended execution order
1. Run Setup and Configuration.
2. Run Ollama Helper and Connection Test.
3. Run Tools, File Saver, and Memory cells.
4. Run the Optimized `run_agent()` cell.
5. Run the Demo and inspect the saved files/memory.
6. Run Automated Tests.
7. Run the five-output batch when ready (this makes five LLM calls).
8. Generate the Streamlit app, README, and report draft.

## 1. Setup and Configuration

In [2]:
import sys
import subprocess
import importlib.util
import json
import re
import time
import sqlite3
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path(r"E:\Agentic AI (TrendPilot) Project")
OUTPUT_DIR = PROJECT_ROOT / "outputs"
DATA_DIR = PROJECT_ROOT / "data"
APP_DIR = PROJECT_ROOT / "app"
TEST_DIR = PROJECT_ROOT / "tests"
REPORT_DIR = PROJECT_ROOT / "report"

for folder in (PROJECT_ROOT, OUTPUT_DIR, DATA_DIR, APP_DIR, TEST_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

REQUIRED_PACKAGES = ["ollama", "streamlit"]
missing_packages = [
    package for package in REQUIRED_PACKAGES
    if importlib.util.find_spec(package) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

import ollama

MODEL_NAME = "qwen2.5:3b"
print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)
print("Model:", MODEL_NAME)
print("Folders and required packages are ready.")

Python: d:\Python\python.exe
Project: E:\Agentic AI (TrendPilot) Project
Model: qwen2.5:3b
Folders and required packages are ready.


## 2. Shared Ollama Helper and Connection Test

In [2]:
def ask_llm(prompt, system="You are TrendPilot, a careful social media content planning assistant.", temperature=0.4):
    """Send one prompt to the locally running Ollama model and return its text."""
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        options={"temperature": temperature}
    )
    return response["message"]["content"].strip()

print("Testing Ollama connection...")
try:
    test_response = ask_llm("Reply with exactly: TrendPilot AI is working.", temperature=0)
    print("Model response:", test_response)
except Exception as exc:
    print("Ollama test failed:", exc)
    print("Check that Ollama is running and `qwen2.5:3b` is available.")

Testing Ollama connection...
Model response: TrendPilot AI is working.


## 3. Content Tools

In [3]:
def generate_ideas(topic, audience, platform="Instagram", count=3):
    prompt = f"""Create {count} specific social media content ideas.
Topic: {topic}
Audience: {audience}
Platform: {platform}
For each idea, include the idea, unique angle, suggested format, and audience value.
Do not claim the ideas are currently trending."""
    return ask_llm(prompt)


def generate_hashtags(topic, platform="Instagram", audience="general audience"):
    prompt = f"""Suggest relevant hashtags for this content.
Topic: {topic}
Platform: {platform}
Audience: {audience}
Group them into broad, niche, and audience-specific hashtags.
Do not claim these hashtags are live trending."""
    return ask_llm(prompt)


def write_caption(topic, audience, platform="Instagram", tone="engaging"):
    prompt = f"""Write one platform-appropriate social media caption.
Topic: {topic}
Audience: {audience}
Platform: {platform}
Tone: {tone}
Include a strong opening, useful message, and natural call to action. Avoid unsupported claims."""
    return ask_llm(prompt)


def generate_reel_script(topic, audience, duration=30, tone="engaging"):
    prompt = f"""Write a natural spoken script for approximately {duration} seconds.
Topic: {topic}
Audience: {audience}
Tone: {tone}
Return four clearly labelled sections: Hook, Main explanation, Visual suggestion, Closing line/CTA.
Do not invent statistics or claim live trend access."""
    return ask_llm(prompt)


def generate_title_thumbnail(topic, platform="Instagram"):
    prompt = f"""Generate three title and thumbnail-text combinations.
Topic: {topic}
Platform: {platform}
For each, include title, short thumbnail_text, and content_angle.
Avoid deceptive clickbait. Thumbnail text is wording only, not an image."""
    return ask_llm(prompt)


def review_content(content, audience, platform):
    prompt = f"""Review this content before publication.
Audience: {audience}
Platform: {platform}
Content:
{content}
Assess clarity, relevance, hook, unsupported claims, and readability. Give a score out of 10, issues, and specific improvements."""
    return ask_llm(prompt)

print("Content tools ready:", [
    "generate_ideas", "generate_hashtags", "write_caption",
    "generate_reel_script", "generate_title_thumbnail", "review_content"
])

Content tools ready: ['generate_ideas', 'generate_hashtags', 'write_caption', 'generate_reel_script', 'generate_title_thumbnail', 'review_content']


## 4. File Saver — TXT, Markdown, and JSON

In [4]:
def save_output(content, filename=None, file_format="md"):
    """Save text, Markdown, or JSON output and return its absolute path."""
    allowed_formats = {"txt", "md", "json"}
    file_format = str(file_format).lower().lstrip(".")
    if file_format not in allowed_formats:
        raise ValueError("Format must be txt, md, or json.")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if filename is None:
        filename = "trendpilot_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    filename = re.sub(r"[^a-zA-Z0-9_-]", "_", str(filename))
    path = OUTPUT_DIR / f"{filename}.{file_format}"

    if file_format == "json":
        if isinstance(content, str):
            try:
                content = json.loads(content)
            except json.JSONDecodeError:
                content = {"content": content}
        path.write_text(json.dumps(content, indent=2, ensure_ascii=False), encoding="utf-8")
    else:
        if not isinstance(content, str):
            content = json.dumps(content, indent=2, ensure_ascii=False)
        path.write_text(content, encoding="utf-8")
    return str(path)

print("File Saver ready:", OUTPUT_DIR)

File Saver ready: E:\Agentic AI (TrendPilot) Project\outputs


## 5. Persistent Memory — SQLite

In [5]:
DB_PATH = DATA_DIR / "trendpilot_memory.db"


def init_memory():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS tasks (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                created_at TEXT NOT NULL,
                topic TEXT NOT NULL,
                audience TEXT,
                platform TEXT,
                result TEXT,
                saved_file TEXT
            )
        """)


def remember_task(topic, audience, platform, result, saved_file):
    """Store the task and the path to its saved JSON result."""
    init_memory()
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            INSERT INTO tasks (created_at, topic, audience, platform, result, saved_file)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (
            datetime.now().isoformat(timespec="seconds"),
            str(topic), str(audience), str(platform), str(result), str(saved_file)
        ))


def get_recent_memory(limit=5):
    limit = max(1, min(int(limit), 100))
    init_memory()
    with sqlite3.connect(DB_PATH) as conn:
        rows = conn.execute("""
            SELECT created_at, topic, audience, platform, saved_file
            FROM tasks ORDER BY id DESC LIMIT ?
        """, (limit,)).fetchall()
    return [
        {"created_at": r[0], "topic": r[1], "audience": r[2],
         "platform": r[3], "saved_file": r[4]}
        for r in rows
    ]


init_memory()
print("SQLite memory ready:", DB_PATH)

SQLite memory ready: E:\Agentic AI (TrendPilot) Project\data\trendpilot_memory.db


## 6. Optimized Agent — Keep This as the Only `run_agent()` Definition

In [6]:
def run_agent(
    topic,
    audience,
    platform="Instagram",
    tone="engaging",
    duration=30,
    create_ideas=True,
    create_caption=True,
    create_hashtags=True,
    create_script=True,
    create_titles=True,
    review=False
):
    """Generate a structured content plan, validate it, save it, and remember it.

    The main generation uses one Ollama call. One repair call is attempted only
    if the structured output fails validation. Optional review uses one extra call.
    """
    start_time = time.perf_counter()
    topic = str(topic).strip()
    audience = str(audience).strip()
    platform = str(platform).strip()
    tone = str(tone).strip()

    if not topic:
        raise ValueError("Topic cannot be empty.")
    if not audience:
        raise ValueError("Target audience cannot be empty.")
    if not platform:
        raise ValueError("Platform cannot be empty.")
    if not tone:
        raise ValueError("Tone cannot be empty.")
    try:
        duration = int(duration)
    except (TypeError, ValueError) as exc:
        raise ValueError("Duration must be an integer between 10 and 180 seconds.") from exc
    if not 10 <= duration <= 180:
        raise ValueError("Duration must be between 10 and 180 seconds.")

    enabled = {
        "ideas": bool(create_ideas),
        "caption": bool(create_caption),
        "hashtags": bool(create_hashtags),
        "reel_script": bool(create_script),
        "titles_thumbnails": bool(create_titles)
    }
    if not any(enabled.values()):
        raise ValueError("Enable at least one content-generation section.")

    requirements = []
    if create_ideas:
        requirements.append("ideas: exactly 3 specific content ideas as a list of strings")
    if create_caption:
        requirements.append("caption: one engaging platform-appropriate caption with a call to action")
    if create_hashtags:
        requirements.append("hashtags: exactly 5 relevant hashtag strings without # symbols; do not claim live trends")
    if create_script:
        requirements.append(
            f"reel_script: object with hook, main_explanation, visual_suggestion, closing_line; "
            f"natural spoken wording for about {duration} seconds"
        )
    if create_titles:
        requirements.append(
            "titles_thumbnails: exactly 3 objects, each with title, thumbnail_text, content_angle; "
            "thumbnail_text is text only, not an image"
        )

    schema = {}
    if create_ideas:
        schema["ideas"] = ["idea 1", "idea 2", "idea 3"]
    if create_caption:
        schema["caption"] = "Caption with a call to action"
    if create_hashtags:
        schema["hashtags"] = ["tag1", "tag2", "tag3", "tag4", "tag5"]
    if create_script:
        schema["reel_script"] = {
            "hook": "Opening hook",
            "main_explanation": "Natural spoken explanation",
            "visual_suggestion": "What viewers should see",
            "closing_line": "Closing line and CTA"
        }
    if create_titles:
        schema["titles_thumbnails"] = [
            {"title": "Specific title", "thumbnail_text": "Short wording", "content_angle": "Content focus"},
            {"title": "Alternative title", "thumbnail_text": "Alternative wording", "content_angle": "Alternative focus"},
            {"title": "Third title", "thumbnail_text": "Third wording", "content_angle": "Third focus"}
        ]

    prompt = f"""You are TrendPilot, a social media content planning agent.

USER INPUT
Topic: {topic}
Target audience: {audience}
Platform: {platform}
Tone: {tone}
Video duration: {duration} seconds

Generate only these sections:
{chr(10).join('- ' + item for item in requirements)}

Expected JSON structure:
{json.dumps(schema, ensure_ascii=False)}

QUALITY RULES
1. Make content specific to the topic and audience.
2. Avoid generic filler and unsupported factual claims.
3. Titles must accurately match the content.
4. Make the script natural when spoken aloud.
5. Keep hooks engaging but honest.
6. Hashtags must be relevant; do not claim they are live trending.
7. Do not pretend to access live trend data.
8. Thumbnail text is text only; do not claim an image was generated.
9. Return valid JSON only, without Markdown fences.
10. Include only requested sections.
"""
    system_prompt = (
        "You are a precise content strategist. Return a valid JSON object only. "
        "Never claim access to live social-media trends."
    )

    def parse_json(response):
        if isinstance(response, dict):
            return response
        text_response = str(response).strip()
        text_response = re.sub(r"^\s*```(?:json)?\s*", "", text_response, flags=re.IGNORECASE)
        text_response = re.sub(r"\s*```\s*$", "", text_response)
        try:
            parsed = json.loads(text_response)
        except json.JSONDecodeError:
            start, end = text_response.find("{"), text_response.rfind("}")
            if start < 0 or end <= start:
                raise ValueError("The model did not return a JSON object.")
            parsed = json.loads(text_response[start:end + 1])
        if not isinstance(parsed, dict):
            raise ValueError("The model output must be a JSON object.")
        return parsed

    def validate_result(data):
        errors = []
        if not isinstance(data, dict):
            return ["Output must be a JSON object."]
        if create_ideas:
            value = data.get("ideas")
            if not isinstance(value, list) or len(value) < 3:
                errors.append("Provide at least 3 content ideas.")
            elif not all(isinstance(x, str) and x.strip() for x in value[:3]):
                errors.append("Each content idea must be a non-empty string.")
        if create_caption:
            value = data.get("caption")
            if not isinstance(value, str) or not value.strip():
                errors.append("Caption must be a non-empty string.")
        if create_hashtags:
            value = data.get("hashtags")
            if not isinstance(value, list) or len(value) < 5:
                errors.append("Provide at least 5 hashtags.")
            elif not all(isinstance(x, str) and x.strip() for x in value[:5]):
                errors.append("Each hashtag must be a non-empty string.")
        if create_script:
            value = data.get("reel_script")
            fields = ["hook", "main_explanation", "visual_suggestion", "closing_line"]
            if not isinstance(value, dict):
                errors.append("Reel script must be an object with four fields.")
            else:
                for field in fields:
                    if not isinstance(value.get(field), str) or not value[field].strip():
                        errors.append(f"Reel script field is missing or empty: {field}.")
        if create_titles:
            value = data.get("titles_thumbnails")
            if not isinstance(value, list) or len(value) < 3:
                errors.append("Provide at least 3 title-thumbnail combinations.")
            else:
                fields = ["title", "thumbnail_text", "content_angle"]
                for index, item in enumerate(value[:3], start=1):
                    if not isinstance(item, dict):
                        errors.append(f"Title combination {index} must be an object.")
                        continue
                    for field in fields:
                        if not isinstance(item.get(field), str) or not item[field].strip():
                            errors.append(f"Title combination {index} has missing or empty {field}.")
        return errors

    print("Generating content with one LLM call...")
    parse_issue = None
    try:
        generated = parse_json(ask_llm(prompt, system=system_prompt))
    except (ValueError, json.JSONDecodeError) as exc:
        generated = {}
        parse_issue = str(exc)

    generated = {key: value for key, value in generated.items() if key in enabled and enabled[key]}
    errors = validate_result(generated)
    if parse_issue:
        errors.insert(0, parse_issue)
    repair_attempted = bool(errors)

    if repair_attempted:
        print("Output needs correction. Attempting one repair call...")
        repair_prompt = f"""Correct this TrendPilot JSON output.
Topic: {topic}
Audience: {audience}
Platform: {platform}
Tone: {tone}
Duration: {duration} seconds
Required sections: {json.dumps(requirements, ensure_ascii=False)}
Validation errors: {json.dumps(errors, ensure_ascii=False)}
Current output: {json.dumps(generated, ensure_ascii=False)}
Return the complete corrected JSON object only. Do not claim live trend access.
"""
        try:
            repaired = parse_json(ask_llm(repair_prompt, system=system_prompt))
            for key, is_enabled in enabled.items():
                if is_enabled and key in repaired:
                    generated[key] = repaired[key]
            errors = validate_result(generated)
        except (ValueError, json.JSONDecodeError) as exc:
            errors.append(f"Repair failed: {exc}")

    generated_at = datetime.now().isoformat(timespec="seconds")
    result = {
        "agent": "TrendPilot",
        "topic": topic,
        "audience": audience,
        "platform": platform,
        "tone": tone,
        "duration_seconds": duration,
        "generated_at": generated_at,
        "plan": requirements,
        **generated,
        "quality_check": {
            "valid": len(errors) == 0,
            "issues": errors,
            "repair_attempted": repair_attempted
        }
    }

    if review:
        print("Reviewing content with one additional LLM call...")
        result["review"] = review_content(json.dumps(result, ensure_ascii=False, indent=2), audience, platform)

    elapsed = round(time.perf_counter() - start_time, 2)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    json_path = OUTPUT_DIR / f"trendpilot_{timestamp}.json"
    md_path = OUTPUT_DIR / f"trendpilot_{timestamp}.md"
    result["saved_json"] = str(json_path)
    result["saved_markdown"] = str(md_path)
    result["saved_file"] = str(json_path)  # Compatibility with batch/test cells
    result["execution_seconds"] = elapsed
    result["memory_saved"] = False

    markdown_lines = [
        f"# TrendPilot Content Plan: {topic}", "",
        f"**Audience:** {audience}", f"**Platform:** {platform}",
        f"**Tone:** {tone}", f"**Generated:** {generated_at}", ""
    ]
    for key in generated:
        value = generated[key]
        markdown_lines.extend([f"## {key.replace('_', ' ').title()}", ""])
        if isinstance(value, (dict, list)):
            markdown_lines.extend(["```json", json.dumps(value, indent=2, ensure_ascii=False), "```"])
        else:
            markdown_lines.append(str(value))
        markdown_lines.append("")
    markdown_lines.extend([
        "## Quality Check", "",
        f"Valid: {result['quality_check']['valid']}",
        f"Repair attempted: {repair_attempted}",
        f"Issues: {json.dumps(errors, ensure_ascii=False)}", ""
    ])
    if review:
        markdown_lines.extend(["## Review", "", str(result["review"]), ""])
    md_path.write_text("\n".join(markdown_lines), encoding="utf-8")

    # Set the expected success state before storing the record. If storage fails,
    # the exception handler resets it and the JSON file is rewritten below.
    try:
        result["memory_saved"] = True
        remember_task(topic, audience, platform, json.dumps(result, ensure_ascii=False), str(json_path))
        print("Task saved to SQLite memory.")
    except Exception as exc:
        result["memory_saved"] = False
        print(f"Memory save failed: {exc}")

    json_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\nSaved JSON: {json_path}")
    print(f"Saved Markdown: {md_path}")
    print(f"Quality valid: {result['quality_check']['valid']}")
    print(f"Repair attempted: {repair_attempted}")
    print(f"Memory saved: {result['memory_saved']}")
    print(f"Completed in {elapsed} seconds")
    return result

print("Latest TrendPilot run_agent() is ready.")

Latest TrendPilot run_agent() is ready.


## 7. Demo Run

In [7]:
demo_result = run_agent(
    topic="AI tools for university students",
    audience="University students and beginners",
    platform="Instagram",
    tone="engaging",
    duration=30,
    review=False
)

print("\n--- TEST SUMMARY ---")
print("Topic:", demo_result["topic"])
print("Quality valid:", demo_result["quality_check"]["valid"])
print("Memory saved:", demo_result["memory_saved"])
print("Execution time:", demo_result["execution_seconds"], "seconds")
print("JSON file:", demo_result["saved_json"])
print("Markdown file:", demo_result["saved_markdown"])

Generating content with one LLM call...
Output needs correction. Attempting one repair call...
Task saved to SQLite memory.

Saved JSON: E:\Agentic AI (TrendPilot) Project\outputs\trendpilot_20261013_170512_878006.json
Saved Markdown: E:\Agentic AI (TrendPilot) Project\outputs\trendpilot_20261013_170512_878006.md
Quality valid: False
Repair attempted: True
Memory saved: True
Completed in 63.53 seconds

--- TEST SUMMARY ---
Topic: AI tools for university students
Quality valid: False
Memory saved: True
Execution time: 63.53 seconds
JSON file: E:\Agentic AI (TrendPilot) Project\outputs\trendpilot_20261013_170512_878006.json
Markdown file: E:\Agentic AI (TrendPilot) Project\outputs\trendpilot_20261013_170512_878006.md


## 8. Inspect Recent Memory and Saved Files

In [8]:
print("Recent memory:")
print(json.dumps(get_recent_memory(10), indent=2, ensure_ascii=False))

print("\nSaved output files:")
for file_path in sorted(OUTPUT_DIR.glob("*"), key=lambda p: p.stat().st_mtime, reverse=True):
    if file_path.is_file():
        print(f"{file_path.name} — {file_path.stat().st_size:,} bytes")

Recent memory:
[
  {
    "created_at": "2026-10-13T17:05:12",
    "topic": "AI tools for university students",
    "audience": "University students and beginners",
    "platform": "Instagram",
    "saved_file": "E:\\Agentic AI (TrendPilot) Project\\outputs\\trendpilot_20261013_170512_878006.json"
  },
  {
    "created_at": "2026-10-13T16:45:12",
    "topic": "AI tools for university students",
    "audience": "University students and beginners",
    "platform": "Instagram",
    "saved_file": "E:\\Agentic AI (TrendPilot) Project\\outputs\\trendpilot_20261013_164512_546671.json"
  },
  {
    "created_at": "2026-10-13T16:44:30",
    "topic": "AI tools for university students",
    "audience": "University students and beginners",
    "platform": "Instagram",
    "saved_file": "E:\\Agentic AI (TrendPilot) Project\\outputs\\trendpilot_20261013_164430_517587.json"
  },
  {
    "created_at": "2026-10-13T16:16:57",
    "topic": "AI tools for university students",
    "audience": "University stu

## 9. Automated Tests — 11 Checks

In [9]:
test_results = []

def record_test(name, passed, details=""):
    test_results.append({
        "test": name,
        "status": "PASS" if passed else "FAIL",
        "details": str(details)
    })

# 1. Project directories
record_test("Project directories", PROJECT_ROOT.exists() and OUTPUT_DIR.exists() and DATA_DIR.exists())

# 2. SQLite database
try:
    init_memory()
    record_test("Memory database initialized", DB_PATH.exists())
except Exception as exc:
    record_test("Memory database initialized", False, exc)

# 3-5. File Saver formats
for fmt, content in [("txt", "Test content"), ("md", "# Test"), ("json", {"test": True})]:
    try:
        path = Path(save_output(content, f"test_{fmt}", fmt))
        record_test(f"File Saver {fmt.upper()}", path.exists() and path.stat().st_size > 0, path)
    except Exception as exc:
        record_test(f"File Saver {fmt.upper()}", False, exc)

# 6. Reject invalid format
try:
    save_output("x", "test_invalid", "pdf")
    record_test("Invalid file format rejected", False, "No ValueError was raised")
except ValueError:
    record_test("Invalid file format rejected", True)

# 7-8. Agent input validation (these must not call the LLM)
for name, kwargs in [
    ("Empty topic rejected", {"topic": "", "audience": "students"}),
    ("Empty audience rejected", {"topic": "AI", "audience": ""})
]:
    try:
        run_agent(**kwargs)
        record_test(name, False, "No ValueError was raised")
    except ValueError as exc:
        record_test(name, True, exc)
    except Exception as exc:
        record_test(name, False, f"Unexpected exception: {exc}")

# 9. Duration validation
try:
    run_agent("AI", "students", duration=5)
    record_test("Invalid duration rejected", False, "No ValueError was raised")
except ValueError:
    record_test("Invalid duration rejected", True)

# 10. Memory retrieval
try:
    memory = get_recent_memory(5)
    record_test("Memory retrieval", isinstance(memory, list))
except Exception as exc:
    record_test("Memory retrieval", False, exc)

# 11. Required tools
required_tools = [generate_ideas, generate_hashtags, write_caption,
                  generate_reel_script, generate_title_thumbnail,
                  review_content, save_output, remember_task, get_recent_memory]
record_test("Required tools are callable", all(callable(tool) for tool in required_tools))

for item in test_results:
    print(f"{item['status']}: {item['test']}" + (f" — {item['details']}" if item['details'] else ""))
passed = sum(item["status"] == "PASS" for item in test_results)
failed = len(test_results) - passed
print(f"\nTotal: {len(test_results)} | Passed: {passed} | Failed: {failed}")
test_report_path = TEST_DIR / "test_results.json"
test_report_path.write_text(json.dumps(test_results, indent=2, ensure_ascii=False), encoding="utf-8")
print("Test report:", test_report_path)

PASS: Project directories
PASS: Memory database initialized
PASS: File Saver TXT — E:\Agentic AI (TrendPilot) Project\outputs\test_txt.txt
PASS: File Saver MD — E:\Agentic AI (TrendPilot) Project\outputs\test_md.md
PASS: File Saver JSON — E:\Agentic AI (TrendPilot) Project\outputs\test_json.json
PASS: Invalid file format rejected
PASS: Empty topic rejected — Topic cannot be empty.
PASS: Empty audience rejected — Target audience cannot be empty.
PASS: Invalid duration rejected
PASS: Memory retrieval
PASS: Required tools are callable

Total: 11 | Passed: 11 | Failed: 0
Test report: E:\Agentic AI (TrendPilot) Project\tests\test_results.json


## 10. Generate Five Saved Content Plans

In [10]:
# This cell makes five LLM calls. Run it when ready; each plan is saved to outputs/.
sample_topics = [
    ("AI tools for students", "University students", "Instagram"),
    ("Beginner Python mistakes", "Programming beginners", "YouTube"),
    ("Healthy study routines", "College students", "TikTok"),
    ("Cybersecurity basics", "Non-technical users", "Instagram"),
    ("How to build a portfolio", "Fresh graduates", "LinkedIn")
]

batch_results = []
for index, (topic, audience, platform) in enumerate(sample_topics, start=1):
    print(f"\nGenerating output {index}/5: {topic}")
    try:
        result = run_agent(
            topic=topic, audience=audience, platform=platform,
            create_ideas=True, create_caption=True, create_hashtags=True,
            create_script=True, create_titles=True, review=False
        )
        batch_results.append({"topic": topic, "status": "SUCCESS", "saved_file": result["saved_file"]})
    except Exception as exc:
        batch_results.append({"topic": topic, "status": "FAILED", "error": str(exc)})

batch_report_path = OUTPUT_DIR / "five_output_batch_results.json"
batch_report_path.write_text(json.dumps(batch_results, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nBatch summary:")
print(json.dumps(batch_results, indent=2, ensure_ascii=False))
print("Batch report:", batch_report_path)


Generating output 1/5: AI tools for students
Generating content with one LLM call...
Output needs correction. Attempting one repair call...
Task saved to SQLite memory.

Saved JSON: E:\Agentic AI (TrendPilot) Project\outputs\trendpilot_20261013_170624_120349.json
Saved Markdown: E:\Agentic AI (TrendPilot) Project\outputs\trendpilot_20261013_170624_120349.md
Quality valid: False
Repair attempted: True
Memory saved: True
Completed in 71.16 seconds

Generating output 2/5: Beginner Python mistakes
Generating content with one LLM call...
Output needs correction. Attempting one repair call...
Task saved to SQLite memory.

Saved JSON: E:\Agentic AI (TrendPilot) Project\outputs\trendpilot_20261013_170745_820303.json
Saved Markdown: E:\Agentic AI (TrendPilot) Project\outputs\trendpilot_20261013_170745_820303.md
Quality valid: False
Repair attempted: True
Memory saved: True
Completed in 81.68 seconds

Generating output 3/5: Healthy study routines
Generating content with one LLM call...
Output n

## 11. Generate the Streamlit App

In [6]:
import json
import re
import sqlite3
from pathlib import Path
from datetime import datetime

import ollama
import streamlit as st

PROJECT_ROOT = Path(r"E:\Agentic AI (TrendPilot) Project")
OUTPUT_DIR = PROJECT_ROOT / "outputs"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = DATA_DIR / "trendpilot_memory.db"
MODEL_NAME = "qwen2.5:3b"

st.set_page_config(page_title="TrendPilot AI", page_icon="🚀", layout="wide")
st.title("🚀 TrendPilot — AI Content Planning Agent")
st.caption("Local Ollama model. Suggestions are not verified live trends.")

def save_and_remember(topic, audience, platform, result):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    json_path = OUTPUT_DIR / f"streamlit_trendpilot_{stamp}.json"
    md_path = OUTPUT_DIR / f"streamlit_trendpilot_{stamp}.md"
    result["saved_json"] = str(json_path)
    result["saved_markdown"] = str(md_path)
    result["saved_file"] = str(json_path)
    json_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    md_content = f"# TrendPilot Content Plan: {topic}\n\n**Audience:** {audience}\n\n**Platform:** {platform}\n\n```json\n" + json.dumps(result, indent=2, ensure_ascii=False) + "\n```\n"
    md_path.write_text(md_content, encoding="utf-8")
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("CREATE TABLE IF NOT EXISTS tasks (id INTEGER PRIMARY KEY AUTOINCREMENT, created_at TEXT NOT NULL, topic TEXT NOT NULL, audience TEXT, platform TEXT, result TEXT, saved_file TEXT)")
        conn.execute("INSERT INTO tasks (created_at, topic, audience, platform, result, saved_file) VALUES (?, ?, ?, ?, ?, ?)",
            (datetime.now().isoformat(timespec="seconds"), topic, audience, platform, json.dumps(result, ensure_ascii=False), str(json_path)))
    return json_path, md_path

topic = st.text_input("Content topic", "AI tools for university students")
audience = st.text_input("Target audience", "University students and beginners")
platform = st.selectbox("Platform", ["Instagram", "TikTok", "YouTube", "LinkedIn"])
tone = st.selectbox("Tone", ["Engaging", "Educational", "Professional", "Friendly", "Humorous"])
duration = st.selectbox("Video duration (seconds)", [15, 30, 45, 60])

if st.button("Generate Content Plan", type="primary"):
    if not topic.strip() or not audience.strip():
        st.error("Please enter both a topic and a target audience.")
    else:
        prompt = (f"Return one valid JSON object for a social media content plan.\n"
                  f"Topic: {topic}\nAudience: {audience}\nPlatform: {platform}\n"
                  f"Tone: {tone}\nVideo duration: {duration} seconds\n"
                  "Include keys: ideas (3 strings), caption (string), hashtags (5 strings without #), "
                  "reel_script (object with hook, main_explanation, visual_suggestion, closing_line), "
                  "titles_thumbnails (3 objects with title, thumbnail_text, content_angle), "
                  "improvement_suggestions (3 strings). Do not claim live trend access. Return JSON only.")
        with st.spinner("Generating with local Ollama model..."):
            try:
                response = ollama.chat(model=MODEL_NAME, messages=[
                    {"role": "system", "content": "You are TrendPilot. Return valid JSON only and never claim live trend access."},
                    {"role": "user", "content": prompt}
                ], options={"temperature": 0.4})
                raw = response["message"]["content"].strip()
                raw = re.sub(r"^\s*```(?:json)?\s*", "", raw, flags=re.IGNORECASE)
                raw = re.sub(r"\s*```\s*$", "", raw)
                try:
                    result = json.loads(raw)
                except json.JSONDecodeError:
                    start, end = raw.find("{"), raw.rfind("}")
                    if start < 0 or end <= start:
                        raise ValueError("Model did not return a JSON object.")
                    result = json.loads(raw[start:end + 1])
                if not isinstance(result, dict):
                    raise ValueError("Model output must be a JSON object.")
                result.update({"topic": topic, "audience": audience, "platform": platform,
                               "tone": tone, "generated_at": datetime.now().isoformat(timespec="seconds")})
                json_path, md_path = save_and_remember(topic, audience, platform, result)
                st.subheader("Generated Content Plan")
                st.json(result)
                st.success("Saved JSON and Markdown; task added to SQLite memory.")
                st.write("JSON file:", str(json_path))
                st.write("Markdown file:", str(md_path))
            except Exception as exc:
                st.error(f"Generation failed: {exc}")
                st.info("Check that Ollama is running and qwen2.5:3b is installed.")


2026-10-15 11:54:22.966 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-15 11:54:22.967 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-15 11:54:22.968 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-15 11:54:22.968 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-15 11:54:22.969 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-15 11:54:22.970 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-15 11:54:22.970 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-15 11:54:22.972 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

## 12. Launch Streamlit Locally

In [7]:
print(f'Run this command in the VS Code terminal:\npython -m streamlit run "{APP_DIR / "streamlit_app.py"}"')

Run this command in the VS Code terminal:
python -m streamlit run "E:\Agentic AI (TrendPilot) Project\app\streamlit_app.py"


## 13. Create README.md

In [13]:
readme = r"""# TrendPilot — Agentic AI Content Planning Agent

TrendPilot is a local-LLM content planning assistant built for the AIRI Team PITB AI Internship Task 2.

## Features
- Content ideas, captions, hashtags, short-video scripts, titles, and thumbnail wording
- Structured JSON output, validation, and one repair attempt
- Optional content review
- File Saver for TXT, Markdown, and JSON
- Persistent SQLite task memory
- Local Streamlit interface using Ollama
- Automated checks and a five-output batch workflow

## Stack
Python, Ollama, Qwen 2.5 3B, SQLite, Streamlit, JSON, Markdown.

## Setup
1. Install Ollama from https://ollama.com/.
2. Download the model: `ollama pull qwen2.5:3b`
3. Install packages: `python -m pip install ollama streamlit`
4. Open `TrendPilot_Updated.ipynb` in VS Code and run cells in order.
5. Run the Streamlit app using the command printed by the notebook.

## Agent workflow
Receive topic/audience/platform -> validate inputs -> generate structured plan -> validate JSON -> optionally repair/review -> save JSON and Markdown -> store task in SQLite -> return result.

## Output and memory
Content plans are saved in `outputs/`. Task history is stored in `data/trendpilot_memory.db`.

## Limitations
TrendPilot does not verify live trends. Model outputs can be inaccurate or repetitive and should be reviewed. The Streamlit interface runs locally and requires Ollama to be available on the same computer.

## Testing
Run the notebook's automated test cell. Report the actual pass/fail results rather than assuming tests passed. Run the five-output batch separately; it makes five model calls.
"""
readme_path = PROJECT_ROOT / "README.md"
readme_path.write_text(readme.strip() + "\n", encoding="utf-8")
print("README created:", readme_path)

README created: E:\Agentic AI (TrendPilot) Project\README.md


## 14. Create Final Report Draft

In [14]:
report_text = r"""# TrendPilot — Final Project Report Draft

## 1. Project Overview
TrendPilot is a local-LLM content-planning agent built with Python, Ollama, Qwen 2.5 3B, SQLite, and Streamlit. It accepts a topic, target audience, platform, tone, and video duration and returns a structured content plan.

## 2. Agent Workflow
1. Validate user inputs.
2. Construct a prompt for the requested sections.
3. Generate structured JSON with the local Ollama model.
4. Validate required sections and attempt one repair if validation fails.
5. Optionally review the content.
6. Save JSON and Markdown files.
7. Store the task and output path in SQLite memory.
8. Return the content plan and execution status.

## 3. Tools
- Idea generator
- Hashtag generator
- Caption writer
- Reel-script generator
- Title and thumbnail-text generator
- Content reviewer
- File Saver (TXT, Markdown, JSON)
- SQLite memory

## 4. Demo Outputs
Insert screenshots of the notebook's successful demo, generated JSON/Markdown, SQLite memory, and Streamlit UI. Add examples from the five-output batch after running it.

## 5. Testing
Insert the actual results from `tests/test_results.json`. Distinguish structural validation from factual accuracy; a valid JSON structure does not prove that every statement is correct.

## 6. Failure Analysis — Complete After Testing
Document five real weak responses or failures. For each one, include: input, observed output/problem, likely cause, correction made, and whether the retest improved. Do not invent failure examples if they were not observed.

## 7. Lessons Learned — Complete Based on Your Work
Write 5–7 lessons from the actual development process, such as structured output validation, local model latency, prompt design, file persistence, memory integration, UI integration, and testing. Keep only lessons that reflect what happened in this project.

## 8. Limitations
- TrendPilot does not access live social-media trend data.
- The local model may produce generic, repetitive, or incorrect suggestions.
- Validation checks structure and required fields, not full factual correctness.
- Performance depends on the local computer and model size.

## 9. Future Improvements
- Connect a reliable trend-data source.
- Improve quality scoring and evaluation.
- Add user preferences and memory-aware personalization.
- Add stronger error handling and automated regression tests.
- Connect the Streamlit UI to the same shared agent module instead of maintaining a separate execution path.

## 10. Suggested LinkedIn Post
I’m excited to share TrendPilot, a local-LLM-powered AI content planning agent developed as part of the AIRI Team PITB AI Internship Task 2. It helps generate structured social media content plans, including ideas, captions, hashtags, short-video scripts, titles, and thumbnail text. The project uses Ollama and Qwen 2.5 3B, with SQLite memory, file export, and a local Streamlit interface. This project helped me practise structured LLM output, validation, persistent storage, and building an AI workflow end to end.

## Appendix
Add the final project structure, test report, screenshots, and five saved output examples.
"""
report_path = REPORT_DIR / "final_report_draft.md"
report_path.write_text(report_text.strip() + "\n", encoding="utf-8")
print("Report draft created:", report_path)

Report draft created: E:\Agentic AI (TrendPilot) Project\report\final_report_draft.md


## 15. Final Project Verification

In [15]:
print("Project root exists:", PROJECT_ROOT.exists())
print("Agent callable:", callable(run_agent))
print("Memory database exists:", DB_PATH.exists())
print("Saved outputs:", len([p for p in OUTPUT_DIR.glob("*") if p.is_file()]))
print("Notebook helper tools callable:", all(callable(f) for f in [
    generate_ideas, generate_hashtags, write_caption, generate_reel_script,
    generate_title_thumbnail, review_content, save_output, remember_task, get_recent_memory
]))
print("Important paths:")
for p in [OUTPUT_DIR, DATA_DIR, APP_DIR, TEST_DIR, REPORT_DIR]:
    print("-", p)

Project root exists: True
Agent callable: True
Memory database exists: True
Saved outputs: 26
Notebook helper tools callable: True
Important paths:
- E:\Agentic AI (TrendPilot) Project\outputs
- E:\Agentic AI (TrendPilot) Project\data
- E:\Agentic AI (TrendPilot) Project\app
- E:\Agentic AI (TrendPilot) Project\tests
- E:\Agentic AI (TrendPilot) Project\report
